# Dataset Validation & Splitting Pipeline

Complete validation and split generation:
1. Load and inspect dataset
2. Validate data integrity
3. Descriptive statistics
4. Class and kinase analysis
5. Kinase-aware grouping for splits
6. Generate TRAIN/VALIDATION/TEST without leakage
7. Export split datasets

In [1]:
import pickle
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')

print("✓ All imports successful")

✓ All imports successful


## Section 1: Load Dataset

In [2]:
BASE_DIR = Path.cwd()
DATASET_PKL = BASE_DIR / "dataset_ready.pkl"

print(f"Loading dataset from: {DATASET_PKL}")

with open(DATASET_PKL, 'rb') as f:
    dataset = pickle.load(f)

print(f"\n✓ Dataset loaded successfully")
print(f"\nTop-level keys: {list(dataset.keys())}")
print(f"\nDataset structure:")
print(f"  - num_samples: {dataset['num_samples']}")
print(f"  - class_distribution: {dataset['class_distribution']}")
print(f"  - kinase_distribution: {dataset['kinase_distribution']}")

# Extract structures
structures = dataset['structures']
print(f"\nTotal structures: {len(structures)}")
print(f"\nFirst structure keys: {list(structures[0].keys())}")
print(f"\nSample structure (first entry):")
s = structures[0]
print(f"  sample_id: {s['sample_id']}")
print(f"  kinase_name: {s['kinase_name']}")
print(f"  conformation_class: {s['conformation_class']}")
print(f"  ca_coords shape: {s['ca_coords'].shape}")
print(f"  adjacency shape: {s['adjacency'].shape}")
print(f"  embedding shape: {s['embedding'].shape}")

Loading dataset from: /home/user/tpf2/vision_avanzada_tpf/dataset_ready.pkl

✓ Dataset loaded successfully

Top-level keys: ['structures', 'num_samples', 'class_distribution', 'kinase_distribution']

Dataset structure:
  - num_samples: 795
  - class_distribution: {'INACTIVE': 424, 'ACTIVE': 368, 'UNKNOWN': 3}
  - kinase_distribution: {'EGFR': 373, 'BRAF': 194, 'ABL1': 49, 'KIT': 44, 'PDGFRA': 10, 'FGFR1': 125}

Total structures: 795

First structure keys: ['sample_id', 'pdb_id', 'kinase_name', 'num_residues', 'ca_coords', 'adjacency', 'num_edges', 'conformation_class', 'sequence', 'embedding']

Sample structure (first entry):
  sample_id: EGFR_3w33_A
  kinase_name: EGFR
  conformation_class: INACTIVE
  ca_coords shape: (297, 3)
  adjacency shape: (297, 297)
  embedding shape: (426, 1280)


## Section 2: Validate Data Integrity

In [3]:
print("Validating data integrity...\n")

validation_issues = []

# Validate each structure
for idx, struct in enumerate(structures):
    sample_id = struct['sample_id']
    
    # Check embeddings
    if struct['embedding'] is None:
        validation_issues.append(f"{sample_id}: embedding is None")
    elif struct['embedding'].shape[1] != 1280:
        validation_issues.append(f"{sample_id}: embedding dim {struct['embedding'].shape[1]} != 1280")
    
    # Check coordinates
    if struct['ca_coords'].shape[1] != 3:
        validation_issues.append(f"{sample_id}: ca_coords has {struct['ca_coords'].shape[1]} columns, expected 3")
    
    # Check adjacency matrix
    adj = struct['adjacency']
    if adj.shape[0] != adj.shape[1]:
        validation_issues.append(f"{sample_id}: adjacency not square ({adj.shape[0]} x {adj.shape[1]})")
    
    # Check symmetry
    if not np.allclose(adj, adj.T):
        validation_issues.append(f"{sample_id}: adjacency not symmetric")
    
    # CRITICAL: Check alignment between coordinates and embeddings
    if struct['ca_coords'].shape[0] != struct['embedding'].shape[0]:
        validation_issues.append(
            f"{sample_id}: CRITICAL - ca_coords ({struct['ca_coords'].shape[0]}) != embedding ({struct['embedding'].shape[0]})"
        )

print(f"Checked {len(structures)} structures\n")

if validation_issues:
    print(f"❌ Found {len(validation_issues)} issues:")
    for issue in validation_issues[:20]:  # Show first 20
        print(f"  - {issue}")
    if len(validation_issues) > 20:
        print(f"  ... and {len(validation_issues) - 20} more")
else:
    print("✓ All structures passed validation")
    print("✓ No data integrity issues found")

# Summary statistics
print(f"\n{'='*80}")
print("VALIDATION SUMMARY:")
print(f"{'='*80}")
print(f"Total structures validated: {len(structures)}")
print(f"Issues found: {len(validation_issues)}")
print(f"Pass rate: {100 * (len(structures) - len(validation_issues)) / len(structures):.1f}%")

Validating data integrity...

Checked 795 structures

❌ Found 760 issues:
  - EGFR_3w33_A: CRITICAL - ca_coords (297) != embedding (426)
  - EGFR_4ll0_B: CRITICAL - ca_coords (291) != embedding (292)
  - EGFR_2ito_A: CRITICAL - ca_coords (303) != embedding (338)
  - EGFR_4li5_A: CRITICAL - ca_coords (306) != embedding (377)
  - EGFR_4jq8_A: CRITICAL - ca_coords (304) != embedding (325)
  - EGFR_3lzb_B: CRITICAL - ca_coords (261) != embedding (283)
  - EGFR_4r5s_A: CRITICAL - ca_coords (299) != embedding (322)
  - EGFR_3vjo_A: CRITICAL - ca_coords (298) != embedding (308)
  - EGFR_3w32_A: CRITICAL - ca_coords (317) != embedding (401)
  - EGFR_3lzb_A: CRITICAL - ca_coords (261) != embedding (283)
  - EGFR_4lqm_A: CRITICAL - ca_coords (305) != embedding (433)
  - EGFR_3w2s_A: CRITICAL - ca_coords (307) != embedding (360)
  - EGFR_2itu_A: CRITICAL - ca_coords (304) != embedding (397)
  - EGFR_4lrm_C: CRITICAL - ca_coords (293) != embedding (296)
  - EGFR_4rix_B: CRITICAL - ca_coords (297) 

## Section 3: Descriptive Statistics

In [4]:
# Extract basic statistics
class_counts = Counter([s['conformation_class'] for s in structures])
kinase_counts = Counter([s['kinase_name'] for s in structures])
residue_counts = [s['ca_coords'].shape[0] for s in structures]
edge_counts = [s['adjacency'].sum() / 2 for s in structures]  # Divide by 2 for undirected
seq_lengths = [len(s['sequence']) for s in structures]

# Class distribution
print("\n1. CLASS DISTRIBUTION:")
print("="*60)
total = len(structures)
for cls in sorted(class_counts.keys()):
    count = class_counts[cls]
    pct = 100 * count / total
    print(f"  {cls:12s}: {count:4d} ({pct:5.1f}%)")

# Kinase distribution
print("\n2. KINASE DISTRIBUTION:")
print("="*60)
for kinase in sorted(kinase_counts.keys()):
    count = kinase_counts[kinase]
    pct = 100 * count / total
    print(f"  {kinase:10s}: {count:4d} ({pct:5.1f}%)")

# Protein size statistics
print("\n3. PROTEIN SIZE STATISTICS (# residues):")
print("="*60)
print(f"  Min:    {min(residue_counts):6d}")
print(f"  Max:    {max(residue_counts):6d}")
print(f"  Mean:   {np.mean(residue_counts):6.1f}")
print(f"  Median: {np.median(residue_counts):6.1f}")
print(f"  Std:    {np.std(residue_counts):6.1f}")

# Graph size statistics
print("\n4. GRAPH SIZE STATISTICS (# edges):")
print("="*60)
print(f"  Min:    {min(edge_counts):6.0f}")
print(f"  Max:    {max(edge_counts):6.0f}")
print(f"  Mean:   {np.mean(edge_counts):6.1f}")
print(f"  Median: {np.median(edge_counts):6.1f}")
print(f"  Std:    {np.std(edge_counts):6.1f}")

# Sequence length statistics
print("\n5. SEQUENCE LENGTH STATISTICS:")
print("="*60)
print(f"  Min:    {min(seq_lengths):6d}")
print(f"  Max:    {max(seq_lengths):6d}")
print(f"  Mean:   {np.mean(seq_lengths):6.1f}")
print(f"  Median: {np.median(seq_lengths):6.1f}")
print(f"  Std:    {np.std(seq_lengths):6.1f}")


1. CLASS DISTRIBUTION:
  ACTIVE      :  368 ( 46.3%)
  INACTIVE    :  424 ( 53.3%)
  UNKNOWN     :    3 (  0.4%)

2. KINASE DISTRIBUTION:
  ABL1      :   49 (  6.2%)
  BRAF      :  194 ( 24.4%)
  EGFR      :  373 ( 46.9%)
  FGFR1     :  125 ( 15.7%)
  KIT       :   44 (  5.5%)
  PDGFRA    :   10 (  1.3%)

3. PROTEIN SIZE STATISTICS (# residues):
  Min:       234
  Max:       614
  Mean:    288.0
  Median:  287.0
  Std:      37.7

4. GRAPH SIZE STATISTICS (# edges):
  Min:      1986
  Max:      5415
  Mean:   2466.8
  Median: 2451.0
  Std:     329.8

5. SEQUENCE LENGTH STATISTICS:
  Min:       248
  Max:       738
  Mean:    357.3
  Median:  329.0
  Std:      89.5


## Section 4: Balance Analysis

In [5]:
# Create crosstab: kinase × class
crosstab_data = []
for struct in structures:
    crosstab_data.append({
        'kinase': struct['kinase_name'],
        'class': struct['conformation_class']
    })

df_cross = pd.DataFrame(crosstab_data)
crosstab = pd.crosstab(df_cross['kinase'], df_cross['class'], margins=True)

print("\nCROSSTAB: Kinase × Conformation Class")
print("="*80)
print(crosstab)

# Calculate ratios
print("\n\nACTIVE/INACTIVE RATIO BY KINASE:")
print("="*80)
for kinase in sorted(kinase_counts.keys()):
    kinase_data = df_cross[df_cross['kinase'] == kinase]
    active = (kinase_data['class'] == 'ACTIVE').sum()
    inactive = (kinase_data['class'] == 'INACTIVE').sum()
    unknown = (kinase_data['class'] == 'UNKNOWN').sum()
    total = len(kinase_data)
    
    if active > 0 and inactive > 0:
        ratio = inactive / active
        print(f"  {kinase:10s}: {active:3d} ACTIVE, {inactive:3d} INACTIVE, {unknown:2d} UNKNOWN | Ratio: {ratio:.2f}")
    else:
        print(f"  {kinase:10s}: {active:3d} ACTIVE, {inactive:3d} INACTIVE, {unknown:2d} UNKNOWN | (Imbalanced)")

# Identify unbalanced kinases
print("\n\nBALANCE ASSESSMENT:")
print("="*80)
for kinase in sorted(kinase_counts.keys()):
    kinase_data = df_cross[df_cross['kinase'] == kinase]
    active = (kinase_data['class'] == 'ACTIVE').sum()
    inactive = (kinase_data['class'] == 'INACTIVE').sum()
    total = len(kinase_data)
    
    if active == 0 or inactive == 0:
        print(f"  ⚠ {kinase:10s}: Missing class (only {active} ACTIVE or {inactive} INACTIVE)")
    elif abs(active - inactive) / total > 0.3:
        print(f"  ⚠ {kinase:10s}: Imbalanced (skew > 30%)")
    else:
        print(f"  ✓ {kinase:10s}: Reasonably balanced")


CROSSTAB: Kinase × Conformation Class
class   ACTIVE  INACTIVE  UNKNOWN  All
kinase                                
ABL1        13        35        1   49
BRAF        45       148        1  194
EGFR       176       196        1  373
FGFR1      121         4        0  125
KIT         11        33        0   44
PDGFRA       2         8        0   10
All        368       424        3  795


ACTIVE/INACTIVE RATIO BY KINASE:
  ABL1      :  13 ACTIVE,  35 INACTIVE,  1 UNKNOWN | Ratio: 2.69
  BRAF      :  45 ACTIVE, 148 INACTIVE,  1 UNKNOWN | Ratio: 3.29
  EGFR      : 176 ACTIVE, 196 INACTIVE,  1 UNKNOWN | Ratio: 1.11
  FGFR1     : 121 ACTIVE,   4 INACTIVE,  0 UNKNOWN | Ratio: 0.03
  KIT       :  11 ACTIVE,  33 INACTIVE,  0 UNKNOWN | Ratio: 3.00
  PDGFRA    :   2 ACTIVE,   8 INACTIVE,  0 UNKNOWN | Ratio: 4.00


BALANCE ASSESSMENT:
  ⚠ ABL1      : Imbalanced (skew > 30%)
  ⚠ BRAF      : Imbalanced (skew > 30%)
  ✓ EGFR      : Reasonably balanced
  ⚠ FGFR1     : Imbalanced (skew > 30%)
  ⚠ KIT

## Section 5: Create Kinase-Aware Splits (TRAIN/VALIDATION/TEST)

In [6]:
print("KINASE-AWARE SPLITTING STRATEGY")
print("="*80)
print("\nObjective: Assign kinases to splits with ZERO leakage between them")
print("Constraint: Each kinase must belong to exactly ONE split")
print("Priority: Maximize class coverage in each split\n")

# Get unique kinases
kinases = sorted(set([s['kinase_name'] for s in structures]))
print(f"Available kinases (n={len(kinases)}): {kinases}\n")

# Get kinase counts and class distributions
kinase_info = {}
for kinase in kinases:
    kinase_structs = [s for s in structures if s['kinase_name'] == kinase]
    active = sum(1 for s in kinase_structs if s['conformation_class'] == 'ACTIVE')
    inactive = sum(1 for s in kinase_structs if s['conformation_class'] == 'INACTIVE')
    unknown = sum(1 for s in kinase_structs if s['conformation_class'] == 'UNKNOWN')
    
    kinase_info[kinase] = {
        'total': len(kinase_structs),
        'active': active,
        'inactive': inactive,
        'unknown': unknown,
        'ratio': inactive / active if active > 0 else float('inf')
    }

# Define splits
# Strategy: Group kinases to balance both sample count AND class distribution
train_kinases = ['EGFR', 'BRAF']  # Large, both have balanced ACTIVE/INACTIVE
validation_kinases = ['ABL1', 'FGFR1']  # Medium size
test_kinases = ['KIT', 'PDGFRA']  # Smaller sets

print("PROPOSED SPLIT ASSIGNMENT:")
print("-"*80)
print(f"\nTRAIN kinases:      {train_kinases}")
for k in train_kinases:
    info = kinase_info[k]
    print(f"  {k:10s}: {info['total']:4d} samples ({info['active']:3d} ACTIVE, {info['inactive']:3d} INACTIVE, {info['unknown']:2d} UNKNOWN)")

print(f"\nVALIDATION kinases: {validation_kinases}")
for k in validation_kinases:
    info = kinase_info[k]
    print(f"  {k:10s}: {info['total']:4d} samples ({info['active']:3d} ACTIVE, {info['inactive']:3d} INACTIVE, {info['unknown']:2d} UNKNOWN)")

print(f"\nTEST kinases:       {test_kinases}")
for k in test_kinases:
    info = kinase_info[k]
    print(f"  {k:10s}: {info['total']:4d} samples ({info['active']:3d} ACTIVE, {info['inactive']:3d} INACTIVE, {info['unknown']:2d} UNKNOWN)")

# Create split datasets
train_data = [s for s in structures if s['kinase_name'] in train_kinases]
validation_data = [s for s in structures if s['kinase_name'] in validation_kinases]
test_data = [s for s in structures if s['kinase_name'] in test_kinases]

print(f"\n{'='*80}")
print(f"SPLIT SIZES: TRAIN={len(train_data)}, VALIDATION={len(validation_data)}, TEST={len(test_data)}")

KINASE-AWARE SPLITTING STRATEGY

Objective: Assign kinases to splits with ZERO leakage between them
Constraint: Each kinase must belong to exactly ONE split
Priority: Maximize class coverage in each split

Available kinases (n=6): ['ABL1', 'BRAF', 'EGFR', 'FGFR1', 'KIT', 'PDGFRA']

PROPOSED SPLIT ASSIGNMENT:
--------------------------------------------------------------------------------

TRAIN kinases:      ['EGFR', 'BRAF']
  EGFR      :  373 samples (176 ACTIVE, 196 INACTIVE,  1 UNKNOWN)
  BRAF      :  194 samples ( 45 ACTIVE, 148 INACTIVE,  1 UNKNOWN)

VALIDATION kinases: ['ABL1', 'FGFR1']
  ABL1      :   49 samples ( 13 ACTIVE,  35 INACTIVE,  1 UNKNOWN)
  FGFR1     :  125 samples (121 ACTIVE,   4 INACTIVE,  0 UNKNOWN)

TEST kinases:       ['KIT', 'PDGFRA']
  KIT       :   44 samples ( 11 ACTIVE,  33 INACTIVE,  0 UNKNOWN)
  PDGFRA    :   10 samples (  2 ACTIVE,   8 INACTIVE,  0 UNKNOWN)

SPLIT SIZES: TRAIN=567, VALIDATION=174, TEST=54


## Section 6: Validate Splits

In [7]:
# Function to get stats for a split
def get_split_stats(data, split_name):
    kinases = set([s['kinase_name'] for s in data])
    active = sum(1 for s in data if s['conformation_class'] == 'ACTIVE')
    inactive = sum(1 for s in data if s['conformation_class'] == 'INACTIVE')
    unknown = sum(1 for s in data if s['conformation_class'] == 'UNKNOWN')
    
    return {
        'name': split_name,
        'num_samples': len(data),
        'kinases': kinases,
        'num_kinases': len(kinases),
        'active': active,
        'inactive': inactive,
        'unknown': unknown,
        'active_pct': 100 * active / len(data) if len(data) > 0 else 0,
        'inactive_pct': 100 * inactive / len(data) if len(data) > 0 else 0,
    }

# Get stats for each split
train_stats = get_split_stats(train_data, "TRAIN")
val_stats = get_split_stats(validation_data, "VALIDATION")
test_stats = get_split_stats(test_data, "TEST")

print("\nSPLIT STATISTICS:")
print("="*80)

for stats in [train_stats, val_stats, test_stats]:
    print(f"\n{stats['name']} SET:")
    print(f"  Samples: {stats['num_samples']}")
    print(f"  Kinases ({stats['num_kinases']}): {sorted(stats['kinases'])}")
    print(f"  Classes: {stats['active']} ACTIVE ({stats['active_pct']:.1f}%), "
          f"{stats['inactive']} INACTIVE ({stats['inactive_pct']:.1f}%), "
          f"{stats['unknown']} UNKNOWN")

# Check for leakage
print(f"\n{'='*80}")
print("LEAKAGE CHECK:")
print("-"*80)

train_kinases_set = train_stats['kinases']
val_kinases_set = val_stats['kinases']
test_kinases_set = test_stats['kinases']

leakage_found = False

# Check pairwise intersections
if train_kinases_set & val_kinases_set:
    print(f"❌ LEAKAGE: TRAIN ∩ VALIDATION = {train_kinases_set & val_kinases_set}")
    leakage_found = True
if train_kinases_set & test_kinases_set:
    print(f"❌ LEAKAGE: TRAIN ∩ TEST = {train_kinases_set & test_kinases_set}")
    leakage_found = True
if val_kinases_set & test_kinases_set:
    print(f"❌ LEAKAGE: VALIDATION ∩ TEST = {val_kinases_set & test_kinases_set}")
    leakage_found = True

if not leakage_found:
    print("✓ NO LEAKAGE DETECTED")
    print("✓ All splits are kinase-disjoint")

# Check coverage
total_coverage = train_stats['num_samples'] + val_stats['num_samples'] + test_stats['num_samples']
print(f"\n{'='*80}")
print("COVERAGE:")
print(f"  Total samples: {len(structures)}")
print(f"  Covered in splits: {total_coverage}")
print(f"  Coverage: {100 * total_coverage / len(structures):.1f}%")


SPLIT STATISTICS:

TRAIN SET:
  Samples: 567
  Kinases (2): ['BRAF', 'EGFR']
  Classes: 221 ACTIVE (39.0%), 344 INACTIVE (60.7%), 2 UNKNOWN

VALIDATION SET:
  Samples: 174
  Kinases (2): ['ABL1', 'FGFR1']
  Classes: 134 ACTIVE (77.0%), 39 INACTIVE (22.4%), 1 UNKNOWN

TEST SET:
  Samples: 54
  Kinases (2): ['KIT', 'PDGFRA']
  Classes: 13 ACTIVE (24.1%), 41 INACTIVE (75.9%), 0 UNKNOWN

LEAKAGE CHECK:
--------------------------------------------------------------------------------
✓ NO LEAKAGE DETECTED
✓ All splits are kinase-disjoint

COVERAGE:
  Total samples: 795
  Covered in splits: 795
  Coverage: 100.0%


## Section 7: Export Split Datasets

In [8]:
# Create split dataset structures maintaining original format
def create_split_dataset(structures_list, split_name):
    """Create a dataset dict following the same structure as dataset_ready.pkl"""
    class_dist = Counter([s['conformation_class'] for s in structures_list])
    kinase_dist = Counter([s['kinase_name'] for s in structures_list])
    
    return {
        'structures': structures_list,
        'num_samples': len(structures_list),
        'class_distribution': dict(class_dist),
        'kinase_distribution': dict(kinase_dist),
        'split_name': split_name,
    }

# Create datasets
train_dataset = create_split_dataset(train_data, 'TRAIN')
validation_dataset = create_split_dataset(validation_data, 'VALIDATION')
test_dataset = create_split_dataset(test_data, 'TEST')

# Export to pickle files
train_pkl = BASE_DIR / "train_dataset.pkl"
validation_pkl = BASE_DIR / "validation_dataset.pkl"
test_pkl = BASE_DIR / "test_dataset.pkl"

print("Exporting split datasets...\n")

with open(train_pkl, 'wb') as f:
    pickle.dump(train_dataset, f)
print(f"✓ {train_pkl.name}: {len(train_data)} samples")

with open(validation_pkl, 'wb') as f:
    pickle.dump(validation_dataset, f)
print(f"✓ {validation_pkl.name}: {len(validation_data)} samples")

with open(test_pkl, 'wb') as f:
    pickle.dump(test_dataset, f)
print(f"✓ {test_pkl.name}: {len(test_data)} samples")

# Verify files exist and check sizes
print(f"\n{'='*80}")
print("FILE VERIFICATION:")
print("-"*80)

for pkl_file in [train_pkl, validation_pkl, test_pkl]:
    if pkl_file.exists():
        size_mb = pkl_file.stat().st_size / (1024**2)
        print(f"✓ {pkl_file.name:30s} ({size_mb:6.1f} MB)")
    else:
        print(f"❌ {pkl_file.name:30s} (NOT FOUND)")

print(f"\n{'='*80}")

Exporting split datasets...

✓ train_dataset.pkl: 567 samples
✓ validation_dataset.pkl: 174 samples
✓ test_dataset.pkl: 54 samples

FILE VERIFICATION:
--------------------------------------------------------------------------------
✓ train_dataset.pkl              ( 826.0 MB)
✓ validation_dataset.pkl         ( 210.6 MB)
✓ test_dataset.pkl               ( 112.1 MB)



## Section 8: Final Report & Summary

In [9]:
print("\n" + "="*80)
print("FINAL VALIDATION & SPLIT REPORT")
print("="*80)

# 1. Dataset Summary
print("\n1. DATASET SUMMARY")
print("-"*80)
print(f"Total samples in original dataset: {len(structures)}")
print(f"Data integrity issues found: {len(validation_issues)}")
print(f"Data pass rate: {100 * (len(structures) - len(validation_issues)) / len(structures):.1f}%")

# 2. Split sizes
print("\n2. SPLIT DISTRIBUTION")
print("-"*80)
total = len(structures)
print(f"TRAIN:      {len(train_data):4d} samples ({100*len(train_data)/total:5.1f}%)")
print(f"VALIDATION: {len(validation_data):4d} samples ({100*len(validation_data)/total:5.1f}%)")
print(f"TEST:       {len(test_data):4d} samples ({100*len(test_data)/total:5.1f}%)")
print(f"TOTAL:      {total:4d} samples")

# 3. Class distribution
print("\n3. CLASS DISTRIBUTION BY SPLIT")
print("-"*80)
for stats in [train_stats, val_stats, test_stats]:
    print(f"{stats['name']:12s}: {stats['active']:4d} ACTIVE ({stats['active_pct']:5.1f}%), "
          f"{stats['inactive']:4d} INACTIVE ({stats['inactive_pct']:5.1f}%), "
          f"{stats['unknown']:2d} UNKNOWN")

# 4. Kinase coverage
print("\n4. KINASE COVERAGE BY SPLIT")
print("-"*80)
print(f"TRAIN:      {sorted(train_stats['kinases'])}")
print(f"VALIDATION: {sorted(val_stats['kinases'])}")
print(f"TEST:       {sorted(test_stats['kinases'])}")

# 5. Leakage verification
print("\n5. LEAKAGE VERIFICATION (CRITICAL)")
print("-"*80)
print(f"TRAIN ∩ VALIDATION: {train_kinases_set & val_kinases_set if train_kinases_set & val_kinases_set else 'EMPTY ✓'}")
print(f"TRAIN ∩ TEST:       {train_kinases_set & test_kinases_set if train_kinases_set & test_kinases_set else 'EMPTY ✓'}")
print(f"VALIDATION ∩ TEST:  {val_kinases_set & test_kinases_set if val_kinases_set & test_kinases_set else 'EMPTY ✓'}")

if not leakage_found:
    print("\n✓✓✓ NO BIOLOGICAL LEAKAGE DETECTED ✓✓✓")
else:
    print("\n❌ LEAKAGE DETECTED - SPLITS INVALID")

# 6. Files exported
print("\n6. EXPORTED FILES")
print("-"*80)
for pkl_file in [train_pkl, validation_pkl, test_pkl]:
    if pkl_file.exists():
        size_mb = pkl_file.stat().st_size / (1024**2)
        print(f"✓ {pkl_file.name:35s} ({size_mb:6.1f} MB)")
    else:
        print(f"❌ {pkl_file.name:35s} (MISSING)")

# 7. Limitations and notes
print("\n7. STATISTICAL LIMITATIONS & NOTES")
print("-"*80)
print("""
✓ With 6 kinases, optimal strategy is kinase-level grouping
✓ Kinase-disjoint splits eliminate data leakage by design
✓ Class imbalance varies by kinase (see Section 4)

⚠ PDGFRA has only 10 samples (smallest kinase)
⚠ Overall ACTIVE/INACTIVE ratio is 368/424 (slight INACTIVE bias)
⚠ UNKNOWN samples (n=3) are minimal but preserved

📊 Recommendation:
   - Use stratified metrics during evaluation (report by kinase)
   - Consider PDGFRA separately due to small sample size
   - Use macro-averaging for overall performance
""")

# 8. Final checkmarks
print("\n8. VALIDATION CHECKLIST")
print("-"*80)
checks = {
    "Data integrity": len(validation_issues) == 0,
    "No missing embeddings": all(s['embedding'] is not None for s in structures),
    "No missing coordinates": all(s['ca_coords'].shape[0] > 0 for s in structures),
    "Consistent dimensions": all(s['ca_coords'].shape[0] == s['embedding'].shape[0] for s in structures),
    "All kinases covered": len(set([s['kinase_name'] for s in structures])) == 6,
    "No leakage in splits": not leakage_found,
    "All split files exported": all(f.exists() for f in [train_pkl, validation_pkl, test_pkl]),
    "100% coverage": len(train_data) + len(validation_data) + len(test_data) == len(structures),
}

all_pass = True
for check, result in checks.items():
    symbol = "✓" if result else "❌"
    print(f"  {symbol} {check}")
    if not result:
        all_pass = False

print("\n" + "="*80)
if all_pass:
    print("✓✓✓ ALL CHECKS PASSED - DATASET READY FOR TRAINING ✓✓✓")
else:
    print("❌ SOME CHECKS FAILED - REVIEW DATASET")
print("="*80)


FINAL VALIDATION & SPLIT REPORT

1. DATASET SUMMARY
--------------------------------------------------------------------------------
Total samples in original dataset: 795
Data integrity issues found: 760
Data pass rate: 4.4%

2. SPLIT DISTRIBUTION
--------------------------------------------------------------------------------
TRAIN:       567 samples ( 71.3%)
VALIDATION:  174 samples ( 21.9%)
TEST:         54 samples (  6.8%)
TOTAL:       795 samples

3. CLASS DISTRIBUTION BY SPLIT
--------------------------------------------------------------------------------
TRAIN       :  221 ACTIVE ( 39.0%),  344 INACTIVE ( 60.7%),  2 UNKNOWN
VALIDATION  :  134 ACTIVE ( 77.0%),   39 INACTIVE ( 22.4%),  1 UNKNOWN
TEST        :   13 ACTIVE ( 24.1%),   41 INACTIVE ( 75.9%),  0 UNKNOWN

4. KINASE COVERAGE BY SPLIT
--------------------------------------------------------------------------------
TRAIN:      ['BRAF', 'EGFR']
VALIDATION: ['ABL1', 'FGFR1']
TEST:       ['KIT', 'PDGFRA']

5. LEAKAGE VERIF

## Section 9: Split Optimization Analysis

Exhaustive exploration of all possible kinase-to-split assignments to find the optimal balance between class distribution and zero leakage.

In [10]:
from itertools import combinations

print("\n" + "="*80)
print("EXHAUSTIVE SPLIT OPTIMIZATION ANALYSIS")
print("="*80)

# Get all unique kinases
all_kinases = list(set([s['kinase_name'] for s in structures]))
all_kinases.sort()

print(f"\nAvailable kinases: {all_kinases}")
print(f"Total combinations to evaluate: {len(list(combinations(range(len(all_kinases)), 2)))}")

# Generate all valid partitions (train, validation, test)
# For 6 kinases, we need to partition them into 3 non-empty groups
valid_partitions = []

# We'll use a simpler approach: for each way to select train kinases,
# partition remaining into validation and test
for train_size in range(1, len(all_kinases)):
    for train_combo in combinations(all_kinases, train_size):
        remaining = [k for k in all_kinases if k not in train_combo]
        # Now partition remaining into validation and test
        for val_size in range(1, len(remaining)):
            for val_combo in combinations(remaining, val_size):
                test_combo = tuple([k for k in remaining if k not in val_combo])
                if len(test_combo) > 0:  # Must have at least one test kinase
                    valid_partitions.append({
                        'train': set(train_combo),
                        'validation': set(val_combo),
                        'test': set(test_combo),
                    })

print(f"Total valid partitions: {len(valid_partitions)}")

# Evaluate each partition
results = []

for partition in valid_partitions:
    # Get structures for each split
    train_structs = [s for s in structures if s['kinase_name'] in partition['train']]
    val_structs = [s for s in structures if s['kinase_name'] in partition['validation']]
    test_structs = [s for s in structures if s['kinase_name'] in partition['test']]
    
    # Calculate stats for each split
    train_active = sum(1 for s in train_structs if s['conformation_class'] == 'ACTIVE')
    train_inactive = sum(1 for s in train_structs if s['conformation_class'] == 'INACTIVE')
    train_unknown = sum(1 for s in train_structs if s['conformation_class'] == 'UNKNOWN')
    train_total = len(train_structs)
    train_active_pct = 100 * train_active / train_total if train_total > 0 else 0
    
    val_active = sum(1 for s in val_structs if s['conformation_class'] == 'ACTIVE')
    val_inactive = sum(1 for s in val_structs if s['conformation_class'] == 'INACTIVE')
    val_unknown = sum(1 for s in val_structs if s['conformation_class'] == 'UNKNOWN')
    val_total = len(val_structs)
    val_active_pct = 100 * val_active / val_total if val_total > 0 else 0
    
    test_active = sum(1 for s in test_structs if s['conformation_class'] == 'ACTIVE')
    test_inactive = sum(1 for s in test_structs if s['conformation_class'] == 'INACTIVE')
    test_unknown = sum(1 for s in test_structs if s['conformation_class'] == 'UNKNOWN')
    test_total = len(test_structs)
    test_active_pct = 100 * test_active / test_total if test_total > 0 else 0
    
    # Calculate balance score
    balance_score = (abs(train_active_pct - val_active_pct) +
                     abs(train_active_pct - test_active_pct) +
                     abs(val_active_pct - test_active_pct))
    
    results.append({
        'train_kinases': sorted(list(partition['train'])),
        'val_kinases': sorted(list(partition['validation'])),
        'test_kinases': sorted(list(partition['test'])),
        'train_samples': train_total,
        'val_samples': val_total,
        'test_samples': test_total,
        'train_active_pct': train_active_pct,
        'val_active_pct': val_active_pct,
        'test_active_pct': test_active_pct,
        'balance_score': balance_score,
    })

# Sort by balance score
results_sorted = sorted(results, key=lambda x: x['balance_score'])

# Create DataFrame for display
df_results = pd.DataFrame([
    {
        'Train': ', '.join(r['train_kinases']),
        'Val': ', '.join(r['val_kinases']),
        'Test': ', '.join(r['test_kinases']),
        'Train_N': r['train_samples'],
        'Val_N': r['val_samples'],
        'Test_N': r['test_samples'],
        'Train_Act%': f"{r['train_active_pct']:.1f}%",
        'Val_Act%': f"{r['val_active_pct']:.1f}%",
        'Test_Act%': f"{r['test_active_pct']:.1f}%",
        'Balance_Score': f"{r['balance_score']:.2f}",
    }
    for r in results_sorted[:15]  # Top 15
])

print(f"\n{'='*80}")
print("TOP 15 CONFIGURATIONS (sorted by balance score)")
print(f"{'='*80}\n")
print(df_results.to_string(index=False))

# Calculate current split score
current_config = {
    'train_kinases': ['BRAF', 'EGFR'],
    'val_kinases': ['ABL1', 'FGFR1'],
    'test_kinases': ['KIT', 'PDGFRA'],
}

# Find current config in results
current_score = None
for r in results:
    if (sorted(r['train_kinases']) == sorted(current_config['train_kinases']) and
        sorted(r['val_kinases']) == sorted(current_config['val_kinases']) and
        sorted(r['test_kinases']) == sorted(current_config['test_kinases'])):
        current_score = r['balance_score']
        break

best_config = results_sorted[0]
best_score = best_config['balance_score']

print(f"\n{'='*80}")
print("COMPARISON: CURRENT vs OPTIMAL")
print(f"{'='*80}")
print(f"\nCurrent split configuration:")
print(f"  Train: {', '.join(sorted(current_config['train_kinases']))}")
print(f"  Validation: {', '.join(sorted(current_config['val_kinases']))}")
print(f"  Test: {', '.join(sorted(current_config['test_kinases']))}")
print(f"  Balance Score: {current_score:.2f}")

print(f"\nBest split configuration:")
print(f"  Train: {', '.join(sorted(best_config['train_kinases']))}")
print(f"  Validation: {', '.join(sorted(best_config['val_kinases']))}")
print(f"  Test: {', '.join(sorted(best_config['test_kinases']))}")
print(f"  Balance Score: {best_score:.2f}")

if current_score is not None:
    improvement = ((current_score - best_score) / current_score) * 100
    print(f"\nImprovement potential: {improvement:.1f}%")
    
    if improvement > 5:
        print(f"\n⚠ Significant improvement possible ({improvement:.1f}%)")
        print(f"   Consider re-splitting with the recommended configuration")
    elif improvement > 0:
        print(f"\n✓ Current split is reasonably close to optimal ({improvement:.1f}% difference)")
    else:
        print(f"\n✓ Current split is already optimal or near-optimal")

# Show details of best configuration
print(f"\n{'='*80}")
print("DETAILED ANALYSIS OF BEST CONFIGURATION")
print(f"{'='*80}")

best_train_structs = [s for s in structures if s['kinase_name'] in best_config['train_kinases']]
best_val_structs = [s for s in structures if s['kinase_name'] in best_config['val_kinases']]
best_test_structs = [s for s in structures if s['kinase_name'] in best_config['test_kinases']]

for split_name, structs, config_key in [
    ('TRAIN', best_train_structs, 'train_kinases'),
    ('VALIDATION', best_val_structs, 'val_kinases'),
    ('TEST', best_test_structs, 'test_kinases')
]:
    active = sum(1 for s in structs if s['conformation_class'] == 'ACTIVE')
    inactive = sum(1 for s in structs if s['conformation_class'] == 'INACTIVE')
    unknown = sum(1 for s in structs if s['conformation_class'] == 'UNKNOWN')
    total = len(structs)
    active_pct = 100 * active / total if total > 0 else 0
    
    print(f"\n{split_name} ({', '.join(sorted(best_config[config_key]))}):")
    print(f"  Samples: {total}")
    print(f"  ACTIVE: {active} ({active_pct:.1f}%)")
    print(f"  INACTIVE: {inactive}")
    print(f"  UNKNOWN: {unknown}")

print(f"\n{'='*80}")


EXHAUSTIVE SPLIT OPTIMIZATION ANALYSIS

Available kinases: ['ABL1', 'BRAF', 'EGFR', 'FGFR1', 'KIT', 'PDGFRA']
Total combinations to evaluate: 15
Total valid partitions: 540

TOP 15 CONFIGURATIONS (sorted by balance score)

                   Train                       Val                      Test  Train_N  Val_N  Test_N Train_Act% Val_Act% Test_Act% Balance_Score
                    ABL1                      EGFR  BRAF, FGFR1, KIT, PDGFRA       49    373     373      26.5%    47.2%     48.0%         42.92
                    ABL1  BRAF, FGFR1, KIT, PDGFRA                      EGFR       49    373     373      26.5%    48.0%     47.2%         42.92
                    EGFR                      ABL1  BRAF, FGFR1, KIT, PDGFRA      373     49     373      47.2%    26.5%     48.0%         42.92
                    EGFR  BRAF, FGFR1, KIT, PDGFRA                      ABL1      373    373      49      47.2%    48.0%     26.5%         42.92
BRAF, FGFR1, KIT, PDGFRA                      ABL1 

## Section 9: Leave-One-Kinase-Out (LOKO) Cross-Validation

Implement a rigorous biological evaluation strategy where each fold uses one complete kinase as test set, with zero leakage.

In [11]:
# Automatically detect all kinases from dataset
all_kinases_loko = sorted(set([s['kinase_name'] for s in structures]))

print(f"Detected kinases for LOKO: {all_kinases_loko}")
print(f"Total kinases: {len(all_kinases_loko)}\n")

# Generate LOKO folds
loko_folds = []

for fold_idx, test_kinase in enumerate(all_kinases_loko, 1):
    # Get test structures (one complete kinase)
    test_structs = [s for s in structures if s['kinase_name'] == test_kinase]
    
    # Get remaining structures (all other kinases for train+val)
    remaining_structs = [s for s in structures if s['kinase_name'] != test_kinase]
    remaining_kinases = sorted(set([s['kinase_name'] for s in remaining_structs]))
    
    # Split remaining_structs into train and validation
    # Use kinase-based grouping: assign half of kinases to train, half to val
    mid_point = len(remaining_kinases) // 2
    train_kinases_fold = remaining_kinases[:mid_point]
    val_kinases_fold = remaining_kinases[mid_point:]
    
    train_structs_fold = [s for s in remaining_structs if s['kinase_name'] in train_kinases_fold]
    val_structs_fold = [s for s in remaining_structs if s['kinase_name'] in val_kinases_fold]
    
    # Create datasets for this fold
    train_dataset_fold = {
        'structures': train_structs_fold,
        'num_samples': len(train_structs_fold),
        'class_distribution': dict(Counter([s['conformation_class'] for s in train_structs_fold])),
        'kinase_distribution': dict(Counter([s['kinase_name'] for s in train_structs_fold])),
    }
    
    val_dataset_fold = {
        'structures': val_structs_fold,
        'num_samples': len(val_structs_fold),
        'class_distribution': dict(Counter([s['conformation_class'] for s in val_structs_fold])),
        'kinase_distribution': dict(Counter([s['kinase_name'] for s in val_structs_fold])),
    }
    
    test_dataset_fold = {
        'structures': test_structs,
        'num_samples': len(test_structs),
        'class_distribution': dict(Counter([s['conformation_class'] for s in test_structs])),
        'kinase_distribution': dict(Counter([s['kinase_name'] for s in test_structs])),
    }
    
    # Store fold information
    fold_info = {
        'fold_id': fold_idx,
        'test_kinase': test_kinase,
        'train_dataset': train_dataset_fold,
        'validation_dataset': val_dataset_fold,
        'test_dataset': test_dataset_fold,
        'train_kinases': train_kinases_fold,
        'validation_kinases': val_kinases_fold,
        'test_kinases': [test_kinase],
    }
    
    loko_folds.append(fold_info)

print(f"Generated {len(loko_folds)} LOKO folds\n")

Detected kinases for LOKO: ['ABL1', 'BRAF', 'EGFR', 'FGFR1', 'KIT', 'PDGFRA']
Total kinases: 6

Generated 6 LOKO folds



In [12]:
# Validate each fold for leakage
print("="*80)
print("LOKO FOLD VALIDATION")
print("="*80)

leakage_issues = []

for fold in loko_folds:
    fold_id = fold['fold_id']
    test_kinase = fold['test_kinase']
    train_kinases = set(fold['train_kinases'])
    val_kinases = set(fold['validation_kinases'])
    test_kinases = set(fold['test_kinases'])
    
    # Check intersections
    if train_kinases & test_kinases:
        leakage_issues.append(f"Fold {fold_id}: TRAIN ∩ TEST = {train_kinases & test_kinases}")
    if val_kinases & test_kinases:
        leakage_issues.append(f"Fold {fold_id}: VAL ∩ TEST = {val_kinases & test_kinases}")
    if train_kinases & val_kinases:
        leakage_issues.append(f"Fold {fold_id}: TRAIN ∩ VAL = {train_kinases & val_kinases}")

if leakage_issues:
    print("❌ LEAKAGE DETECTED:")
    for issue in leakage_issues:
        print(f"  {issue}")
    raise ValueError("Leakage found in LOKO folds")
else:
    print("✓ NO LEAKAGE DETECTED IN ANY FOLD")
    print("✓ All folds are kinase-disjoint\n")

# Print detailed stats for each fold
for fold in loko_folds:
    fold_id = fold['fold_id']
    test_kinase = fold['test_kinase']
    train_kinases = fold['train_kinases']
    val_kinases = fold['validation_kinases']
    
    train_samples = fold['train_dataset']['num_samples']
    val_samples = fold['validation_dataset']['num_samples']
    test_samples = fold['test_dataset']['num_samples']
    
    train_classes = fold['train_dataset']['class_distribution']
    val_classes = fold['validation_dataset']['class_distribution']
    test_classes = fold['test_dataset']['class_distribution']
    
    print(f"{'='*80}")
    print(f"FOLD {fold_id}")
    print(f"{'='*80}\n")
    
    print(f"TEST KINASE: {test_kinase}\n")
    
    print(f"TRAIN KINASES: {train_kinases}")
    print(f"  Samples: {train_samples}")
    print(f"  ACTIVE: {train_classes.get('ACTIVE', 0)}, INACTIVE: {train_classes.get('INACTIVE', 0)}, UNKNOWN: {train_classes.get('UNKNOWN', 0)}\n")
    
    print(f"VALIDATION KINASES: {val_kinases}")
    print(f"  Samples: {val_samples}")
    print(f"  ACTIVE: {val_classes.get('ACTIVE', 0)}, INACTIVE: {val_classes.get('INACTIVE', 0)}, UNKNOWN: {val_classes.get('UNKNOWN', 0)}\n")
    
    print(f"TEST KINASES: {[test_kinase]}")
    print(f"  Samples: {test_samples}")
    print(f"  ACTIVE: {test_classes.get('ACTIVE', 0)}, INACTIVE: {test_classes.get('INACTIVE', 0)}, UNKNOWN: {test_classes.get('UNKNOWN', 0)}\n")
    
    print(f"Leakage verification:")
    print(f"  TRAIN ∩ TEST: {set(train_kinases) & set([test_kinase]) or 'EMPTY ✓'}")
    print(f"  VAL ∩ TEST: {set(val_kinases) & set([test_kinase]) or 'EMPTY ✓'}")
    print(f"  TRAIN ∩ VAL: {set(train_kinases) & set(val_kinases) or 'EMPTY ✓'}\n")

LOKO FOLD VALIDATION
✓ NO LEAKAGE DETECTED IN ANY FOLD
✓ All folds are kinase-disjoint

FOLD 1

TEST KINASE: ABL1

TRAIN KINASES: ['BRAF', 'EGFR']
  Samples: 567
  ACTIVE: 221, INACTIVE: 344, UNKNOWN: 2

VALIDATION KINASES: ['FGFR1', 'KIT', 'PDGFRA']
  Samples: 179
  ACTIVE: 134, INACTIVE: 45, UNKNOWN: 0

TEST KINASES: ['ABL1']
  Samples: 49
  ACTIVE: 13, INACTIVE: 35, UNKNOWN: 1

Leakage verification:
  TRAIN ∩ TEST: EMPTY ✓
  VAL ∩ TEST: EMPTY ✓
  TRAIN ∩ VAL: EMPTY ✓

FOLD 2

TEST KINASE: BRAF

TRAIN KINASES: ['ABL1', 'EGFR']
  Samples: 422
  ACTIVE: 189, INACTIVE: 231, UNKNOWN: 2

VALIDATION KINASES: ['FGFR1', 'KIT', 'PDGFRA']
  Samples: 179
  ACTIVE: 134, INACTIVE: 45, UNKNOWN: 0

TEST KINASES: ['BRAF']
  Samples: 194
  ACTIVE: 45, INACTIVE: 148, UNKNOWN: 1

Leakage verification:
  TRAIN ∩ TEST: EMPTY ✓
  VAL ∩ TEST: EMPTY ✓
  TRAIN ∩ VAL: EMPTY ✓

FOLD 3

TEST KINASE: EGFR

TRAIN KINASES: ['ABL1', 'BRAF']
  Samples: 243
  ACTIVE: 58, INACTIVE: 183, UNKNOWN: 2

VALIDATION KINASES:

In [13]:
# Create summary table
print("="*80)
print("LOKO SUMMARY TABLE")
print("="*80 + "\n")

summary_data = []
for fold in loko_folds:
    summary_data.append({
        'Fold': fold['fold_id'],
        'Test Kinase': fold['test_kinase'],
        'Train Kinases': ', '.join(fold['train_kinases']),
        'Val Kinases': ', '.join(fold['validation_kinases']),
        'Train Samples': fold['train_dataset']['num_samples'],
        'Val Samples': fold['validation_dataset']['num_samples'],
        'Test Samples': fold['test_dataset']['num_samples'],
    })

df_loko_summary = pd.DataFrame(summary_data)
print(df_loko_summary.to_string(index=False))

print(f"\n\nTotal folds generated: {len(loko_folds)}")
print(f"Total samples covered: {sum(fold['train_dataset']['num_samples'] + fold['validation_dataset']['num_samples'] + fold['test_dataset']['num_samples'] for fold in loko_folds) // 3}")

# Create LOKO folds directory
loko_dir = BASE_DIR / "data" / "loko_folds"
loko_dir.mkdir(parents=True, exist_ok=True)

print(f"\n{'='*80}")
print("EXPORTING LOKO FOLDS")
print("="*80 + "\n")

# Export each fold
for fold in loko_folds:
    fold_id = fold['fold_id']
    test_kinase = fold['test_kinase']
    
    fold_file = loko_dir / f"fold_{fold_id:02d}.pkl"
    
    # Add statistics
    fold_with_stats = fold.copy()
    fold_with_stats['statistics'] = {
        'train_samples': fold['train_dataset']['num_samples'],
        'validation_samples': fold['validation_dataset']['num_samples'],
        'test_samples': fold['test_dataset']['num_samples'],
        'train_classes': fold['train_dataset']['class_distribution'],
        'validation_classes': fold['validation_dataset']['class_distribution'],
        'test_classes': fold['test_dataset']['class_distribution'],
    }
    
    with open(fold_file, 'wb') as f:
        pickle.dump(fold_with_stats, f)
    
    size_mb = fold_file.stat().st_size / (1024**2)
    print(f"✓ {fold_file.name}: {size_mb:.1f} MB (Test kinase: {test_kinase})")

print(f"\n{'='*80}")
print("FILE VERIFICATION")
print("="*80 + "\n")

fold_files = sorted(loko_dir.glob("fold_*.pkl"))
print(f"✓ Created {len(fold_files)} fold files in {loko_dir}\n")

for fold_file in fold_files:
    if fold_file.exists():
        size_mb = fold_file.stat().st_size / (1024**2)
        print(f"✓ {fold_file.name}: {size_mb:6.1f} MB")
    else:
        print(f"❌ {fold_file.name}: NOT FOUND")

print(f"\n{'='*80}")

LOKO SUMMARY TABLE

 Fold Test Kinase Train Kinases         Val Kinases  Train Samples  Val Samples  Test Samples
    1        ABL1    BRAF, EGFR  FGFR1, KIT, PDGFRA            567          179            49
    2        BRAF    ABL1, EGFR  FGFR1, KIT, PDGFRA            422          179           194
    3        EGFR    ABL1, BRAF  FGFR1, KIT, PDGFRA            243          179           373
    4       FGFR1    ABL1, BRAF   EGFR, KIT, PDGFRA            243          427           125
    5         KIT    ABL1, BRAF EGFR, FGFR1, PDGFRA            243          508            44
    6      PDGFRA    ABL1, BRAF    EGFR, FGFR1, KIT            243          542            10


Total folds generated: 6
Total samples covered: 1590

EXPORTING LOKO FOLDS

✓ fold_01.pkl: 1148.8 MB (Test kinase: ABL1)
✓ fold_02.pkl: 1148.8 MB (Test kinase: BRAF)
✓ fold_03.pkl: 1148.8 MB (Test kinase: EGFR)
✓ fold_04.pkl: 1148.8 MB (Test kinase: FGFR1)
✓ fold_05.pkl: 1148.8 MB (Test kinase: KIT)
✓ fold_06.pkl: 1148

In [14]:
# Final LOKO validation checklist
print("LOKO CROSS-VALIDATION FINAL CHECKLIST")
print("="*80 + "\n")

loko_checks = {
    "Kinases detected automatically": len(all_kinases_loko) > 0,
    "Folds generated (one per kinase)": len(loko_folds) == len(all_kinases_loko),
    "All kinases covered as test": set([f['test_kinase'] for f in loko_folds]) == set(all_kinases_loko),
    "No leakage in any fold": not leakage_issues,
    "All folds have train data": all(f['train_dataset']['num_samples'] > 0 for f in loko_folds),
    "All folds have validation data": all(f['validation_dataset']['num_samples'] > 0 for f in loko_folds),
    "All folds have test data": all(f['test_dataset']['num_samples'] > 0 for f in loko_folds),
    "All fold files created": len(fold_files) == len(loko_folds),
    "All fold files valid": all(f.exists() and f.stat().st_size > 0 for f in fold_files),
}

loko_all_pass = True
for check, result in loko_checks.items():
    symbol = "✓" if result else "❌"
    print(f"  {symbol} {check}")
    if not result:
        loko_all_pass = False

print(f"\n{'='*80}")
if loko_all_pass:
    print("✓✓✓ LOKO CROSS-VALIDATION READY FOR USE ✓✓✓")
else:
    print("❌ SOME LOKO CHECKS FAILED")
print(f"{'='*80}\n")

# Summary statistics
print("LOKO SUMMARY STATISTICS")
print("-"*80)
print(f"Total kinases: {len(all_kinases_loko)}")
print(f"Total folds: {len(loko_folds)}")
print(f"LOKO folds directory: {loko_dir}")
print(f"Folds exported: {len(fold_files)}")

total_train = sum(f['train_dataset']['num_samples'] for f in loko_folds)
total_val = sum(f['validation_dataset']['num_samples'] for f in loko_folds)
total_test = sum(f['test_dataset']['num_samples'] for f in loko_folds)

print(f"\nTotal samples across all folds:")
print(f"  Train: {total_train}")
print(f"  Validation: {total_val}")
print(f"  Test: {total_test}")
print(f"  Total: {total_train + total_val + total_test}")

# Show which kinase is used as test in each fold
print(f"\nTest kinase by fold:")
for fold in loko_folds:
    print(f"  Fold {fold['fold_id']}: {fold['test_kinase']}")

LOKO CROSS-VALIDATION FINAL CHECKLIST

  ✓ Kinases detected automatically
  ✓ Folds generated (one per kinase)
  ✓ All kinases covered as test
  ✓ No leakage in any fold
  ✓ All folds have train data
  ✓ All folds have validation data
  ✓ All folds have test data
  ✓ All fold files created
  ✓ All fold files valid

✓✓✓ LOKO CROSS-VALIDATION READY FOR USE ✓✓✓

LOKO SUMMARY STATISTICS
--------------------------------------------------------------------------------
Total kinases: 6
Total folds: 6
LOKO folds directory: /home/user/tpf2/vision_avanzada_tpf/data/loko_folds
Folds exported: 6

Total samples across all folds:
  Train: 1961
  Validation: 2014
  Test: 795
  Total: 4770

Test kinase by fold:
  Fold 1: ABL1
  Fold 2: BRAF
  Fold 3: EGFR
  Fold 4: FGFR1
  Fold 5: KIT
  Fold 6: PDGFRA
